# CS26012 — Groundnut Disease Model Training, run 2 (TensorFlow / Keras, MobileNetV2)

Same pipeline as the first groundnut notebook. **Changes from run 1 are marked 🔁** — everything else is unchanged.
All saved files end in `_model_2`, so run 1's files in Drive are **not overwritten** and the two runs can be compared.

| Part | What it does |
|---|---|
| A | Setup: Drive, repo, libraries, settings |
| B | Explore the dataset 🔁 *(auto-detects whether the folder is already split into Training/Validation/Testing)* |
| C | Duplicate check: exact (MD5) + near-duplicate (perceptual hash) 🔁 *(now also catches flipped/rotated copies)* |
| D | **Clean**: remove cross-class label conflicts + greedy near-duplicate removal |
| E | **Split** 70/15/15 (stratified, seed 42) 🔁 *(group-aware: augmented copies of one photo stay in one split)* → manifest → copy locally |
| F | `tf.data` pipelines 🔁 *(reshuffles every epoch)* + augmentation + class weights |
| G | Build MobileNetV2 (frozen backbone, new head) and train |
| H | Test-set evaluation (in-distribution) |
| I | Save model + metadata to Drive |
| J | Reload check + out-of-distribution (OOD) check on your own photos |

Runtime → Change runtime type → **T4 GPU**.

## A. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/Jugal1406/CS26012-Crop_Disease_Detection.git

In [ ]:
%cd CS26012-Crop_Disease_Detection

In [ ]:
!ls models

In [ ]:
!pip install -q tensorflow scikit-learn matplotlib seaborn pillow imagehash

In [ ]:
import os
import re
import json
import random
import shutil
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
import imagehash

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow Version :", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

### Settings
Identical to run 1 except the new `RUN_TAG`, which is appended to every file this notebook saves. `CROP` must match the value the frontend sends — the backend uses it to pick the model file.

In [ ]:
CROP = 'groundnut'
RUN_TAG = 'model_2'                  # every saved file ends with this -> run 1's files stay untouched

DRIVE_DATA_ROOT = '/content/drive/MyDrive/Crop_Disease_Dataset'
CROP_FOLDERS = {
    'chili':     'Chili Leaf Disease Original Dataset',
    'potato':    'Potato',
    'groundnut': 'Groundnut',
}
LOCAL_DATA_DIR = Path('/content/data') / CROP            # fast local copy (wiped when the session ends)
MODEL_SAVE_DIR = Path('/content/drive/MyDrive/CS26012_models')

SEED          = 42
SPLIT         = (0.70, 0.15, 0.15)   # train / valid / test
DUP_THRESHOLD = 5                    # pHash Hamming distance <= this counts as a near-duplicate
IMG_SIZE      = 224
BATCH_SIZE    = 32
HEAD_EPOCHS   = 30                   # phase 1: backbone frozen (early stopping may end it sooner)
HEAD_LR       = 1e-3
FINETUNE_EPOCHS      = 0             # phase 2 (optional experiment): unfreeze the top of the backbone
FINETUNE_LR          = 1e-5
FINETUNE_LAST_LAYERS = 30

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

## B. Explore the dataset 🔁

**Why this part changed:** this part is now shared with the potato notebook. Groundnut is `Groundnut/<class>/*.jpg` (not pre-split), so it behaves exactly as in run 1. Some datasets ship **already split** as `<Crop>/Training/<class>/…`, `<Crop>/Validation/<class>/…`, `<Crop>/Testing/<class>/…`. The groundnut code would then treat `Training`, `Validation`, `Testing` as three "diseases" — silently wrong.

So first we print the folder tree, then detect the layout:
* **Not pre-split** → class = the folder directly under `<Crop>/` (same as groundnut).
* **Pre-split** → class = the folder under each split folder. All images are **pooled** and we re-do our own clean + 70/15/15 split, so every crop follows the same documented policy as the other crops (and the source's split may itself leak duplicates between train and test).

Every image is collected into one table `img_df` (path, class, filename), and the later cells use that table instead of walking folders again.

In [ ]:
DATASET_DIR = Path(DRIVE_DATA_ROOT) / CROP_FOLDERS[CROP]
print("Dataset exists:", DATASET_DIR.exists())
print("Dataset path:", DATASET_DIR)

def count_images(folder):
    return sum(1 for f in folder.rglob('*') if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS)

# Two-level folder tree, so you can SEE the layout before trusting the auto-detection below
for top in sorted(DATASET_DIR.iterdir()):
    if top.is_dir():
        print(f'{top.name}/  ({count_images(top)} images)')
        for sub in sorted(s for s in top.iterdir() if s.is_dir()):
            print(f'    {sub.name}/  ({count_images(sub)} images)')
    else:
        print(f'{top.name}  (file)')

In [ ]:
SPLIT_FOLDER_NAMES = {'train', 'training', 'valid', 'validation', 'val', 'test', 'testing'}
top_dirs  = [d for d in DATASET_DIR.iterdir() if d.is_dir()]
PRE_SPLIT = len(top_dirs) > 0 and all(d.name.strip().lower() in SPLIT_FOLDER_NAMES for d in top_dirs)
print('Dataset is already split into train/valid/test folders:', PRE_SPLIT)

rows, skipped = [], []
for p in sorted(DATASET_DIR.rglob('*')):
    if not p.is_file() or p.suffix.lower() not in IMAGE_EXTENSIONS:
        continue
    parts = p.relative_to(DATASET_DIR).parts      # e.g. ('Training', 'Early_Blight', 'img1.jpg')
    if PRE_SPLIT:
        if len(parts) < 3: skipped.append(str(p)); continue    # image lying loose in a split folder
        rows.append({'path': str(p), 'class': parts[1].strip(), 'source_split': parts[0], 'filename': p.name})
    else:
        if len(parts) < 2: skipped.append(str(p)); continue    # image lying loose in the crop folder
        rows.append({'path': str(p), 'class': parts[0].strip(), 'source_split': '-', 'filename': p.name})

img_df = pd.DataFrame(rows)
print('Images found:', len(img_df), '| skipped (not inside a class folder):', len(skipped))

# Each class should appear in every source split with the SAME spelling.
# If you see a class twice with different spelling/case (e.g. 'Late_Blight' and 'late blight'),
# rename the folder in Drive and re-run from here.
print(pd.crosstab(img_df['class'], img_df['source_split'], margins=True))

In [ ]:
class_dirs_names = sorted(img_df['class'].unique())
print("Number of classes:", len(class_dirs_names))
for i, c in enumerate(class_dirs_names):
    print(i, "->", c)

class_counts = img_df['class'].value_counts().to_dict()
class_df = (pd.DataFrame(list(class_counts.items()), columns=["Class", "Image_Count"])
              .sort_values("Image_Count", ascending=False).reset_index(drop=True))
class_df

In [ ]:
plt.figure(figsize=(12, 4))
sns.barplot(data=class_df, x="Image_Count", y="Class")
plt.title(f'{CROP.title()} Dataset Class Distribution')
plt.xlabel("Number of Images"); plt.ylabel("Disease Class")
plt.show()

In [ ]:
corrupted_images = []
image_info = []

for path, cls, fname in zip(img_df['path'], img_df['class'], img_df['filename']):
    try:
        with Image.open(path) as img:
            img.verify()                       # checks the file is a valid image
        with Image.open(path) as img:          # verify() leaves the file unusable, so re-open for size
            image_info.append({"class": cls, "filename": fname,
                               "width": img.width, "height": img.height, "mode": img.mode})
    except Exception:
        corrupted_images.append(path)

print("Corrupted images:", len(corrupted_images))
for image in corrupted_images[:20]:
    print(image)

# Drop corrupt files so later steps never crash on them
img_df = img_df[~img_df['path'].isin(corrupted_images)].reset_index(drop=True)

image_info_df = pd.DataFrame(image_info)
print("Images inspected:", len(image_info_df))
image_info_df[["width", "height", "mode"]].describe(include="all")

## C. Duplicate check
**Exact duplicates (MD5):** byte-for-byte identical files. If two identical files carry *different* class labels, the model is being taught a contradiction. (With a pre-split source, identical files in `Training` and `Testing` are also common — they'd inflate the test score.)

In [ ]:
def get_file_hash(file_path):
    md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            md5.update(chunk)
    return md5.hexdigest()

img_df['md5'] = [get_file_hash(p) for p in img_df['path']]

duplicate_groups = img_df.groupby('md5').filter(lambda x: len(x) > 1).sort_values('md5')
print("Images involved in exact-duplicate groups:", len(duplicate_groups))
print("  groups whose copies carry DIFFERENT labels:",
      int((duplicate_groups.groupby('md5')['class'].nunique() > 1).sum()))
if PRE_SPLIT and len(duplicate_groups):
    print("  groups spread over more than one source split:",
          int((duplicate_groups.groupby('md5')['source_split'].nunique() > 1).sum()))

In [ ]:
# Show up to 5 exact-duplicate groups
for i, image_hash in enumerate(duplicate_groups["md5"].unique()[:5]):
    group = duplicate_groups[duplicate_groups["md5"] == image_hash]
    plt.figure(figsize=(12, 4))
    for j, (_, row) in enumerate(group.iterrows()):
        plt.subplot(1, len(group), j + 1)
        plt.imshow(Image.open(row["path"]))
        plt.title(f"{row['class']}\n{row['source_split']}", fontsize=8)
        plt.axis("off")
    plt.suptitle(f"Duplicate Group {i + 1}")
    plt.show()

**Near-duplicates (perceptual hash) 🔁**

pHash turns each image into a 64-bit fingerprint of its overall structure; the *Hamming distance* (how many bits differ) is small when images look alike, even if resized/re-compressed/slightly edited.

**What changed:** plain pHash is **blind to flips and 90° rotations** — a mirrored copy of a leaf gets a completely different fingerprint. Many public datasets (including this groundnut one — its `…_jpg.rf.<hash>` filenames are a Roboflow export) contain offline-augmented flipped/rotated copies. If one copy lands in train and its mirror in test, the test score is inflated.

So for each image we also hash its 8 orientations (original, 3 rotations, mirror, mirror + 3 rotations). The distance between two images is the **smallest** distance over those orientations. `phash` (orientation 0) is still the same fingerprint as before.

Memory note: the all-vs-all comparison needs about N² × 8 bytes temporarily (≈ 0.6 GB for groundnut's ~8,500 images) — fine on Colab.

In [ ]:
ORIENTATIONS = [None,
                Image.Transpose.ROTATE_90, Image.Transpose.ROTATE_180, Image.Transpose.ROTATE_270,
                Image.Transpose.FLIP_LEFT_RIGHT, Image.Transpose.TRANSPOSE,
                Image.Transpose.FLIP_TOP_BOTTOM, Image.Transpose.TRANSVERSE]

def orientation_phashes(img):
    # pHash internally converts to grayscale and shrinks to 32x32 — do that once, then flip/rotate
    # the tiny version (cheap). Orientation 0 gives exactly imagehash.phash(img).
    small = img.convert('L').resize((32, 32), Image.LANCZOS)
    return [str(imagehash.phash(small if t is None else small.transpose(t))) for t in ORIENTATIONS]

variants = []
for p in img_df['path']:
    with Image.open(p) as img:
        variants.append(orientation_phashes(img))

df_hash = img_df.copy()
df_hash['phash_variants'] = variants
df_hash['phash'] = [v[0] for v in variants]
print("Total images processed:", len(df_hash))

In [ ]:
def hashes_to_uint64(hex_hashes):
    # '8f3a...'(16 hex chars) -> one 64-bit integer, so numpy can compare them fast
    return np.array([int(h, 16) for h in hex_hashes], dtype=np.uint64)

def variants_to_uint64(variant_lists):
    # list of 8 hex strings per image -> array of shape (N, 8)
    return np.array([[int(h, 16) for h in v] for v in variant_lists], dtype=np.uint64)

def hamming_matrix(df):
    # distance[i, j] = smallest number of differing bits between image i (in any of its 8
    # orientations) and image j (as stored). Made symmetric at the end.
    base = hashes_to_uint64(df['phash'])
    var  = variants_to_uint64(df['phash_variants'])
    dist = np.full((len(df), len(df)), 64, dtype=np.uint8)
    for k in range(var.shape[1]):
        np.minimum(dist, np.bitwise_count(var[:, k][:, None] ^ base[None, :]).astype(np.uint8), out=dist)
    return np.minimum(dist, dist.T)

def near_duplicate_pairs(df, threshold):
    df = df.reset_index(drop=True)
    dist = hamming_matrix(df)
    i, j = np.where(np.triu(dist <= threshold, k=1))       # upper triangle: each pair once, no self-pairs
    col = lambda name, idx: df[name].to_numpy(dtype=object)[idx]
    pairs = pd.DataFrame({
        'image1_class': col('class', i), 'image1': col('filename', i), 'image1_path': col('path', i),
        'image2_class': col('class', j), 'image2': col('filename', j), 'image2_path': col('path', j),
        'hash_distance': dist[i, j],
    })
    pairs['same_class'] = pairs['image1_class'] == pairs['image2_class']
    return pairs

In [ ]:
near_duplicates_df = near_duplicate_pairs(df_hash, DUP_THRESHOLD)
print('Near-duplicate pairs found:', len(near_duplicates_df))
print('  within the same class :', near_duplicates_df['same_class'].sum())
print('  ACROSS two classes    :', (~near_duplicates_df['same_class']).sum(), ' <- label conflicts')

print('\nPairs by distance (0 = identical fingerprint):')
print(near_duplicates_df['hash_distance'].value_counts().sort_index().to_string())

print('\nCross-class pairs by class combination:')
cross = near_duplicates_df[~near_duplicates_df['same_class']]
print(cross.groupby(['image1_class', 'image2_class']).size().to_string() if len(cross) else '  none')

**Look before you trust the threshold.** The cell below shows random pairs at each distance (pairs may be flipped/rotated versions of each other — that's intended). If pairs at distance 5 are clearly *different* leaves, lower `DUP_THRESHOLD` in Settings and re-run from there; if pairs at 5 are still obviously the same photo, you could raise it.

In [ ]:
def show_pairs(pairs, distance, n=3):
    sub = pairs[pairs['hash_distance'] == distance]
    if len(sub) == 0:
        print(f'No pairs at distance {distance}'); return
    sub = sub.sample(min(n, len(sub)), random_state=SEED)
    fig, axes = plt.subplots(len(sub), 2, figsize=(7, 3.3 * len(sub)), squeeze=False)
    for row, (_, p) in enumerate(sub.iterrows()):
        for col, side in enumerate(('image1', 'image2')):
            axes[row][col].imshow(Image.open(p[f'{side}_path']))
            axes[row][col].set_title(f"{p[f'{side}_class']}\n{p[side]}", fontsize=8)
            axes[row][col].axis('off')
    fig.suptitle(f'Hamming distance = {distance}'); plt.tight_layout(); plt.show()

for d in sorted(near_duplicates_df['hash_distance'].unique()):
    show_pairs(near_duplicates_df, d)

## D. Clean the dataset

Same two rules as run 1, applied in this order:

1. **Cross-class conflicts → remove every copy.** If (near-)identical images sit in two different classes, we can't know which label is right, so none of them are trusted.
2. **Within-class near-duplicates → keep one, drop the rest — using the *greedy* method.**
   Images are walked through in sorted path order. Each image is compared **only against images already kept**. If it's within `DUP_THRESHOLD` of a kept image it's dropped; otherwise it becomes a new kept reference.

   ⚠️ This is the method validated earlier in the project. We deliberately do **not** group images into connected clusters (A~B, B~C ⇒ A,B,C merged): that "chaining" wrongly merged dissimilar images at scale.

Only difference: comparisons use the orientation-aware distance from Part C, and images are ordered by full **path** (with a pre-split source, the same filename can exist in two split folders, so filename alone isn't a unique sort key).

Nothing is deleted from Drive — we only build a list (`clean_df`) of which files to use, and save the removed ones to a CSV.

In [ ]:
def find_cross_class_conflicts(df, threshold):
    # Indices of images that have a near-duplicate in a DIFFERENT class.
    dist = hamming_matrix(df)
    labels = df['class'].to_numpy(dtype=object)
    different_class = labels[:, None] != labels[None, :]
    conflict = ((dist <= threshold) & different_class).any(axis=1)
    return set(np.where(conflict)[0])

def greedy_dedup(df, threshold):
    # Returns (kept_indices, {dropped_index: index_of_the_kept_image_it_duplicates}).
    # Works class by class; each image is compared only with already-kept images,
    # in all 8 orientations.
    kept, dropped = [], {}
    for _, group in df.groupby('class', sort=True):
        group = group.sort_values('path')
        ref_idx, ref_hash = [], np.empty(0, dtype=np.uint64)
        for idx, var in zip(group.index, variants_to_uint64(group['phash_variants'])):
            if len(ref_hash):
                d = np.bitwise_count(ref_hash[:, None] ^ var[None, :]).min(axis=1)   # best orientation per reference
                nearest = int(np.argmin(d))
                if d[nearest] <= threshold:
                    dropped[idx] = ref_idx[nearest]
                    continue
            ref_idx.append(idx)
            ref_hash = np.append(ref_hash, var[0])        # store the reference as-is (orientation 0)
        kept.extend(ref_idx)
    return kept, dropped

def clean_dataset(df, threshold):
    df = df.reset_index(drop=True)
    conflicts = find_cross_class_conflicts(df, threshold)
    no_conflict = df.drop(index=list(conflicts))
    kept, dropped = greedy_dedup(no_conflict, threshold)
    clean = df.loc[sorted(kept)].reset_index(drop=True)
    removed = pd.concat([
        df.loc[sorted(conflicts)].assign(reason='cross-class conflict', duplicate_of=''),
        df.loc[list(dropped)].assign(reason='within-class near-duplicate',
                                     duplicate_of=[df.loc[r, 'path'] for r in dropped.values()]),
    ], ignore_index=True)
    return clean, removed

In [ ]:
clean_df, removed_df = clean_dataset(df_hash, DUP_THRESHOLD)

summary = pd.DataFrame({
    'original': df_hash['class'].value_counts(),
    'cross_class_removed': removed_df[removed_df.reason == 'cross-class conflict']['class'].value_counts(),
    'near_dup_removed': removed_df[removed_df.reason == 'within-class near-duplicate']['class'].value_counts(),
    'clean': clean_df['class'].value_counts(),
}).fillna(0).astype(int).sort_index()
print(summary.to_string())
print(f"\nTotal: {len(df_hash)} -> {len(clean_df)} images ({len(removed_df)} removed)")
print(f"Max:min ratio after cleaning = {summary['clean'].max() / summary['clean'].min():.1f}:1 (policy target <= 5:1)")
for c, n in summary['clean'].items():
    if n < 100: print(f'  WARNING: "{c}" has only {n} images (hard floor is 100)')

MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
removed_df.drop(columns=['phash_variants']).to_csv(MODEL_SAVE_DIR / f'{CROP}_removed_images_{RUN_TAG}.csv', index=False)
print('Removed-image log saved to', MODEL_SAVE_DIR / f'{CROP}_removed_images_{RUN_TAG}.csv')

## E. Split 70 / 15 / 15 and copy locally 🔁

Same as run 1 — stratified, seed 42, after de-duplication, manifest saved, files **copied** to local disk — plus two fixes:

* **Group-aware split.** Images exported by tools like Roboflow are named `<original>_jpg.rf.<random>.jpg`; several files with the same `<original>` are augmented copies of **one photo**. Augmentations such as arbitrary-angle rotation, crop or shear can slip past pHash, so we also group files by that original name and put each **whole group** into one split. If the dataset has no such names, every image is its own group and this behaves exactly like the groundnut split.
* **Unique local filenames.** When a pre-split source is pooled, two different images can share a filename within one class (e.g. `1.jpg` in both `Training/` and `Testing/`). The run-1 copy step would silently overwrite one. Each copy now gets a numeric prefix.

In [ ]:
ROBOFLOW_SUFFIX = re.compile(r'[._-](jpe?g|png|bmp|webp)\.rf\.[0-9a-f]+$', re.IGNORECASE)

def source_group(cls, filename):
    # 'Early-Leaf-Spot-104-_jpg.rf.e8ced1da...jpg' -> 'groundnut_early_leaf_spot/Early-Leaf-Spot-104-'
    return f'{cls}/{ROBOFLOW_SUFFIX.sub("", Path(filename).stem)}'

clean_df['group'] = [source_group(c, f) for c, f in zip(clean_df['class'], clean_df['filename'])]
group_sizes = clean_df.groupby('group').size()
print(f'{len(clean_df)} images come from {len(group_sizes)} distinct source photos')
print(f'Groups with more than one file (augmented copies of one photo): {(group_sizes > 1).sum()}')
print(clean_df.groupby('class')['group'].nunique().rename('distinct source photos').to_string())

In [ ]:
def stratified_group_split(df, fractions=(0.7, 0.15, 0.15), seed=42):
    rng = random.Random(seed)
    parts = []
    for cls, g in df.groupby('class', sort=True):
        groups = sorted(g['group'].unique())
        rng.shuffle(groups)
        n = len(groups)
        n_valid, n_test = int(n * fractions[1]), int(n * fractions[2])
        n_train = n - n_valid - n_test                     # rounding leftovers go to train
        assign = {**{x: 'train' for x in groups[:n_train]},
                  **{x: 'valid' for x in groups[n_train:n_train + n_valid]},
                  **{x: 'test'  for x in groups[n_train + n_valid:]}}
        parts.append(g.assign(split=g['group'].map(assign))[['path', 'class', 'group', 'split']])
    return pd.concat(parts, ignore_index=True)

def copy_split(manifest, dest_root):
    dest_root = Path(dest_root)
    if dest_root.exists():
        shutil.rmtree(dest_root)          # start clean so re-runs never mix old + new files
    for i, (path, cls, split) in enumerate(zip(manifest['path'], manifest['class'], manifest['split'])):
        target = dest_root / split / cls
        target.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target / f'{i:06d}_{Path(path).name}')   # prefix -> never overwrite

In [ ]:
manifest = stratified_group_split(clean_df, SPLIT, SEED)

# Safety check: no source photo may appear in two splits
leaks = manifest.groupby('group')['split'].nunique()
assert (leaks == 1).all(), f'{(leaks > 1).sum()} groups appear in more than one split'

manifest.to_csv(MODEL_SAVE_DIR / f'{CROP}_split_manifest_{RUN_TAG}.csv', index=False)
print(manifest.pivot_table(index='class', columns='split', values='path', aggfunc='count')[['train', 'valid', 'test']])

copy_split(manifest, LOCAL_DATA_DIR)
n_copied = sum(1 for _ in LOCAL_DATA_DIR.rglob('*') if _.is_file())
print(f'\nCopied {n_copied} files to {LOCAL_DATA_DIR} (manifest has {len(manifest)})')
assert n_copied == len(manifest)

## F. Input pipeline, augmentation, class weights 🔁

Same as run 1, with one fix to the order of `cache()` and shuffling.

In run 1, `image_dataset_from_directory(shuffle=True)` shuffled **once**, and `.cache()` placed after it froze that order — so every epoch saw identical batches in identical order (augmentation still varied, so the damage was small). Here images are loaded unbatched → cached → **reshuffled every epoch** → batched.

* Valid/test are **not** shuffled, so predictions line up with labels during evaluation.
* **Augmentation** (random flips, rotation, zoom, contrast, brightness) — Keras layers inside the model; active only during `fit()`.
* **Class weights**: weight = `total / (num_classes × class_count)`, so a rare class's mistakes count more.

In [ ]:
CLASS_NAMES = sorted(clean_df['class'].unique())
NUM_CLASSES = len(CLASS_NAMES)
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(split, shuffle):
    ds = keras.utils.image_dataset_from_directory(
        str(LOCAL_DATA_DIR / split),
        labels='inferred', label_mode='int', class_names=CLASS_NAMES,
        image_size=(IMG_SIZE, IMG_SIZE), batch_size=None,    # unbatched, so we can shuffle per image
        shuffle=False,
    )
    ds = ds.cache()                                          # decoded images kept in RAM after epoch 1
    if shuffle:
        ds = ds.shuffle(4096, seed=SEED, reshuffle_each_iteration=True)   # new order every epoch
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_ds('train', shuffle=True)
valid_ds = make_ds('valid', shuffle=False)
test_ds  = make_ds('test',  shuffle=False)

train_counts = manifest[manifest.split == 'train']['class'].value_counts()
class_weight = {i: len(manifest[manifest.split == 'train']) / (NUM_CLASSES * train_counts[c])
                for i, c in enumerate(CLASS_NAMES)}
for i, c in enumerate(CLASS_NAMES):
    print(f'{i}  {c:<30} train={train_counts[c]:<5} weight={class_weight[i]:.2f}')

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.08),          # up to ±0.08 of a full turn ≈ ±29°
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),         # works on 0-255 pixels, so it goes BEFORE rescaling
], name='augmentation')

# Preview: one training image, augmented 8 different ways
for images, labels in train_ds.take(1):
    plt.figure(figsize=(10, 5))
    for k in range(8):
        aug = data_augmentation(images[:1], training=True)
        plt.subplot(2, 4, k + 1); plt.imshow(np.clip(aug[0].numpy(), 0, 255).astype('uint8')); plt.axis('off')
    plt.suptitle(f'Augmentations of one "{CLASS_NAMES[labels[0]]}" image'); plt.show()

## G. Build the model and train
Unchanged from run 1.
```
raw image (0-255) → augmentation → rescale to [-1, 1] → MobileNetV2 backbone (frozen) → average pool → dropout → Dense(softmax)
```
* `Rescaling(1/127.5, offset=-1)` inside the model → the backend feeds plain 0–255 RGB.
* `base.trainable = False` freezes the pretrained backbone; `base(x, training=False)` keeps BatchNorm in inference mode.
* `ModelCheckpoint` keeps the **best** validation epoch; `EarlyStopping` stops after 5 epochs without improvement.

In [ ]:
def build_model():
    base = keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                                          include_top=False, weights='imagenet')
    base.trainable = False
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image_0_255')
    x = data_augmentation(inputs)
    x = layers.Rescaling(1 / 127.5, offset=-1, name='to_minus1_plus1')(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='disease_probs')(x)
    return keras.Model(inputs, outputs, name=f'{CROP}_mobilenetv2'), base

model, base = build_model()
model.compile(optimizer=keras.optimizers.Adam(HEAD_LR),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary(show_trainable=True)

In [ ]:
import time
BEST_PATH = f'/content/{CROP}_best_{RUN_TAG}.keras'
checkpoint_cb = keras.callbacks.ModelCheckpoint(BEST_PATH, monitor='val_accuracy',
                                                save_best_only=True, verbose=1)

start = time.time()
history = model.fit(train_ds, validation_data=valid_ds, epochs=HEAD_EPOCHS,
                    class_weight=class_weight,
                    callbacks=[checkpoint_cb,
                               keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5)])
hist = {k: list(v) for k, v in history.history.items()}

if FINETUNE_EPOCHS > 0:
    # Unfreeze only the top FINETUNE_LAST_LAYERS layers of the backbone. BatchNorm stays in
    # inference mode because the backbone is still called with training=False.
    base.trainable = True
    for layer in base.layers[:-FINETUNE_LAST_LAYERS]:
        layer.trainable = False
    model.compile(optimizer=keras.optimizers.Adam(FINETUNE_LR),   # must re-compile after changing trainable
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    ft = model.fit(train_ds, validation_data=valid_ds,
                   epochs=len(hist['loss']) + FINETUNE_EPOCHS, initial_epoch=len(hist['loss']),
                   class_weight=class_weight,
                   callbacks=[checkpoint_cb,          # same instance -> still remembers the best so far
                              keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5)])
    for k in hist: hist[k] += ft.history.get(k, [])

train_time = time.time() - start
best_epoch = int(np.argmax(hist['val_accuracy'])) + 1
print(f"\nBest val accuracy {max(hist['val_accuracy']):.3f} at epoch {best_epoch} | {train_time:.0f}s")

# Load the best checkpoint (this also proves the saved file loads correctly)
model = keras.models.load_model(BEST_PATH)

In [ ]:
ep = range(1, len(hist['loss']) + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(ep, hist['loss'], label='train'); ax[0].plot(ep, hist['val_loss'], label='valid')
ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(ep, hist['accuracy'], label='train'); ax[1].plot(ep, hist['val_accuracy'], label='valid')
ax[1].set_title('Accuracy'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.show()

## H. Test-set evaluation (in-distribution)
The test set was never used for training *or* for picking the best epoch → this is the honest held-out score.
* **Classification report** — per class: *precision* (when it predicts a disease, how often is it right?), *recall* (of all real leaves with that disease, how many did it catch?), *F1* (balance of both).
* **Confusion matrix** — rows = true class, columns = predicted. For groundnut, watch early ↔ late leaf spot and early rust ↔ rust (run 1's weakest classes were the two leaf spots: F1 0.941 / 0.955).

Compare with run 1 (test acc 0.973). A **lower** number here is not a worse model — it means run 1's score was inflated by flipped/rotated copies leaking between train and test.

⚠️ This number is **in-distribution only** (same source, same camera conditions). Part J measures the other side.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_prob = model.predict(test_ds, verbose=0)
y_pred = y_prob.argmax(axis=1)
test_acc = float((y_pred == y_true).mean())
print(f'TEST accuracy: {test_acc:.3f}  ({len(y_true)} images)\n')
print(classification_report(y_true, y_pred, labels=range(NUM_CLASSES), target_names=CLASS_NAMES, digits=3, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, xticks_rotation=45, cmap='Greens', colorbar=False)
plt.title(f'{CROP} — test confusion matrix'); plt.tight_layout(); plt.show()

## I. Save to Drive
Files written to `MyDrive/CS26012_models/` (all end in `_model_2`; run 1's files are left alone):
* `<crop>_mobilenetv2_model_2.keras` — the full model. If you keep this run, download it into the repo's `models/` folder (rename it, or point the backend at the new name).
* `<crop>_mobilenetv2_model_2_meta.json` — what the **backend** needs: class names in label order, image size, how to prepare an image.
* `<crop>_mobilenetv2_model_2_metrics.json` — numbers for the report/README.
* (already saved earlier) `<crop>_removed_images_model_2.csv`, `<crop>_split_manifest_model_2.csv`.

In [ ]:
import datetime
stem = f'{CROP}_mobilenetv2_{RUN_TAG}'
model.save(str(MODEL_SAVE_DIR / f'{stem}.keras'))

meta = {
    'crop': CROP, 'run': RUN_TAG, 'arch': 'MobileNetV2', 'framework': f'tensorflow {tf.__version__}',
    'class_names': CLASS_NAMES, 'img_size': IMG_SIZE,
    'input': 'RGB, float32, pixel values 0-255, shape (1, 224, 224, 3); the model rescales internally',
    'resize': 'center-crop to a square, then resize to img_size (bilinear)',
}
(MODEL_SAVE_DIR / f'{stem}_meta.json').write_text(json.dumps(meta, indent=2))

metrics = {
    **meta, 'trained_at': datetime.datetime.now().isoformat(timespec='seconds'),
    'seed': SEED, 'split': SPLIT, 'dup_threshold': DUP_THRESHOLD, 'batch_size': BATCH_SIZE,
    'head_epochs': HEAD_EPOCHS, 'head_lr': HEAD_LR, 'finetune_epochs': FINETUNE_EPOCHS, 'finetune_lr': FINETUNE_LR,
    'source_pre_split': PRE_SPLIT, 'dedup': 'greedy pHash, orientation-aware (8 flips/rotations)',
    'split_method': 'stratified, group-aware (augmented copies of one source photo kept in one split)',
    'images_original': len(df_hash), 'images_clean': len(clean_df), 'images_removed': len(removed_df),
    'distinct_source_photos': int(clean_df['group'].nunique()),
    'split_counts': manifest.groupby(['split', 'class']).size().unstack(fill_value=0).to_dict(orient='index'),
    'class_weight': {CLASS_NAMES[i]: round(w, 3) for i, w in class_weight.items()},
    'best_epoch': best_epoch, 'best_val_acc': float(max(hist['val_accuracy'])), 'test_acc': test_acc,
    'train_time_s': round(train_time, 1), 'history': {k: [float(v) for v in vals] for k, vals in hist.items()},
    'test_report': classification_report(y_true, y_pred, labels=range(NUM_CLASSES), target_names=CLASS_NAMES,
                                         output_dict=True, zero_division=0),
    'confusion_matrix': cm.tolist(),
}
to_native = lambda o: o.item() if hasattr(o, 'item') else str(o)   # numpy numbers -> plain Python for JSON
(MODEL_SAVE_DIR / f'{stem}_metrics.json').write_text(json.dumps(metrics, indent=2, default=to_native))

size_mb = (MODEL_SAVE_DIR / f'{stem}.keras').stat().st_size / 1e6
print(f'Saved {stem}.keras ({size_mb:.1f} MB), {stem}_meta.json, {stem}_metrics.json -> {MODEL_SAVE_DIR}')

## J. Reload check + out-of-distribution (OOD) check

`load_for_inference` + `predict_image` are written the way the FastAPI backend will use them: load the `.keras` file and its meta JSON, then for any photo → RGB → center-crop to square → resize to 224 → predict.

**Center-crop first** because phone photos are rarely square; squashing a 4:3 photo would distort the leaf.

Then upload leaf photos **from outside the dataset** (your phone, Google Images, a farm visit). Aim for **at least 10 per class** — one or two photos can't support a number in the report. Record how many are correct, next to the test accuracy.

In [ ]:
def load_for_inference(stem_path):
    m = keras.models.load_model(f'{stem_path}.keras')
    info = json.loads(Path(f'{stem_path}_meta.json').read_text())
    return m, info

def prepare_image(img, size):
    img = img.convert('RGB')
    w, h = img.size
    side = min(w, h)
    left, top = (w - side) // 2, (h - side) // 2
    img = img.crop((left, top, left + side, top + side)).resize((size, size), Image.BILINEAR)
    return np.asarray(img, dtype=np.float32)[None, ...]     # shape (1, size, size, 3), values 0-255

def predict_image(m, info, img, top_k=3):
    probs = m.predict(prepare_image(img, info['img_size']), verbose=0)[0]
    order = probs.argsort()[::-1][:top_k]
    return [(info['class_names'][i], round(float(probs[i]) * 100, 1)) for i in order]

reloaded, info = load_for_inference(MODEL_SAVE_DIR / stem)
same = np.allclose(reloaded.predict(test_ds, verbose=0), y_prob, atol=1e-5)   # same weights -> same outputs
print('Reloaded model matches in-memory predictions:', same)
sample = manifest[manifest.split == 'test'].iloc[0]
print(f"Example: true = {sample['class']} | predicted = {predict_image(reloaded, info, Image.open(sample['path']))}")

In [ ]:
from google.colab import files
uploaded = files.upload()          # pick outside-the-dataset leaf photos (aim for 10+ per class)
for name in uploaded:
    print(f'{name}: {predict_image(reloaded, info, Image.open(name))}')